In [1]:
!jupyter nbconvert --to script FEM_Solver.ipynb
!jupyter nbconvert --to script MeshGenerator.ipynb

[NbConvertApp] Converting notebook FEM_Solver.ipynb to script
[NbConvertApp] Writing 8287 bytes to FEM_Solver.py
[NbConvertApp] Converting notebook MeshGenerator.ipynb to script
[NbConvertApp] Writing 11706 bytes to MeshGenerator.py


In [2]:
# only run if navailable in the environment
# !pip uninstall -y gmsh
# !pip install gmsh
# !pip install scikit-fem

Found existing installation: gmsh 4.15.2
Uninstalling gmsh-4.15.2:
  Successfully uninstalled gmsh-4.15.2
  Using cached gmsh-4.15.2-py2.py3-none-manylinux_2_24_x86_64.whl.metadata (1.7 kB)
Using cached gmsh-4.15.2-py2.py3-none-manylinux_2_24_x86_64.whl (40.0 MB)


In [12]:
from FEM_Solver import compute_P_eff
from MeshGenerator import Simulator
import random
import os
import pandas as pd
import numpy as np
import time
random.seed(7)

# Generate 200 dps:
csv_path = "simulation_200.csv"
design_space = []
modes = ["baseline", "cracks", "debond", "mixed"]

for _ in range(200):
    current_mode = random.choice(modes)
    
    micro_config = {
        "mode": current_mode,
        "fiber_vf": random.uniform(0.1, 0.3),
        "f_r": random.uniform(0.3, 0.8),
        "mu": random.uniform(0.3, 0.8),
        "sigma": random.uniform(1.0, 2.5),
        "db_t": random.uniform(0.05, 0.15),
        # Toggle damage arrays based on selected mode:
        "crc_no": random.randint(2, 15) if current_mode in ["cracks", "mixed"] else 0,
        "debond_fraction": random.uniform(0.1, 0.8) if current_mode in ["debond", "mixed"] else 0.0,
        "base_angle": random.uniform(0.0, np.pi)
    }
    design_space.append(micro_config)

# Run mesh generation
for run_idx, config in enumerate(design_space):
    # check if dp is already generated:
    if os.path.exists(csv_path):
        existing_df = pd.read_csv(csv_path)
        if run_idx in existing_df['run_idx'].values:
            print(f"Skipping run_idx {run_idx}: Already calculated.")
            continue

    start_time = time.time()
    run_config = config.copy()
    # remoces mode, leaving the remainder for simulator
    current_mode = run_config.pop("mode")
    # Initialise simulator with the relevant configurations
    
    sim = Simulator(**run_config)
    pts, tris, dims = sim.run_diffusion(current_mode)
    
    # Run Peff
    P_eff = compute_P_eff(pts, tris, dims)

    row_dp = {
        "run_idx": run_idx,
        "mode": current_mode,
        "fiber_vf": run_config["fiber_vf"],
        "f_r": run_config["f_r"],
        "mu": run_config["mu"],
        "sigma": run_config["sigma"],
        "db_t": run_config["db_t"],
        "crc_no": run_config["crc_no"],
        "debond_fraction": run_config["debond_fraction"],
        "base_angle": run_config["base_angle"],
        "P_eff": P_eff
    }
    
    # Append row to the CSV file
    df_row = pd.DataFrame([row_dp])
    file_exists = os.path.exists(csv_path)
    df_row.to_csv(csv_path, mode='a', index=False, header=not file_exists)

    elapsed = time.time() - start_time
    print(f"DP {run_idx} complete | Time: {elapsed:.2f}s")
    
print("Data generation complete!")

DP 0 complete | Time: 16.86s                                                                                
DP 1 complete | Time: 67.74s                                                                                             
DP 2 complete | Time: 31.49s Making faces                                                                                
DP 3 complete | Time: 30.74s                                                                                             
DP 4 complete | Time: 60.58s                                                                                
DP 5 complete | Time: 9.90s                                                                                                            
DP 6 complete | Time: 10.26s                                                                                                           
DP 7 complete | Time: 11.85s                                                                                                           
DP 8 com